In [1]:
from catalyst.debug.compiler_functions import get_compilation_stage
import pennylane as qml
from catalyst.third_party.oqd import OQDDevice

import os
import shutil
import pathlib
import numpy as np

from functools import partial

########################################################################################

for f in os.listdir():
    if f.startswith("oqd_circuit_qec") and os.path.isdir(f):
        shutil.rmtree(pathlib.Path(f))

compile_results = pathlib.Path("oqd_circuit_qec")
openapl_file_name = "oqd_circuit_qec.openapl.json"


toml_files = {
    "device-toml-loc": "/home/user/oqd-catalyst/scripts/calibration_data/device.toml",
    "qubit-toml-loc": "/home/user/oqd-catalyst/scripts/calibration_data/qubit.toml",
    "gate-to-pulse-toml-loc": "/home/user/oqd-catalyst/scripts/calibration_data/gate.toml",
}

toml_files = " ".join([f"{k}={v}" for k, v in toml_files.items()])


OQD_PIPELINES = [
    (
        "DeviceAgnosticPipeline",
        [
            "quantum-compilation-stage",
            "hlo-lowering-stage",
            "gradient-lowering-stage",
            "bufferization-stage",
        ],
    ),
    (
        "IonDecompositionStage",
        [
            "func.func(ions-decomposition)",
            "func.func(merge-rotations)",
            # "func.func(prune-zero-rotations)",
        ],
    ),
    (
        "IonDialectLoweringStage",
        [
            f"func.func(gates-to-pulses{{{toml_files}}})",
        ],
    ),
    ("IonToLLVMDialectConversion", ["convert-ion-to-llvm"]),
    ("MLIRToLLVMDialectConversion", ["llvm-dialect-lowering-stage"]),
]


oqd_dev = OQDDevice(
    backend="default",
    wires=26,
    openapl_file_name=(compile_results / openapl_file_name).as_posix(),
)


with open("rotated_surface_code.qasm", "r") as f:
    qasm_string = f.read()


@qml.set_shots(10)
@qml.qnode(oqd_dev)
def oqd_circuit_qec():
    qml.from_qasm(qasm_string)()
    return qml.counts(wires=0)


QJIT_CIRCUIT = qml.qjit(
    oqd_circuit_qec, pipelines=OQD_PIPELINES, keep_intermediate=True, verbose=True
)

print("{:=^100}".format("\033[1;32m Compiled circuit \033[0m"))
print(get_compilation_stage(QJIT_CIRCUIT, stage="IonDialectLoweringStage"))


/home/user/oqd-catalyst/.venv/lib/python3.12/site-packages/pennylane_qiskit/converter.py:585: UserWarning: pennylane_qiskit.converter: The Reset instruction is not supported by PennyLane, and has not been added to the template.
  warnings.warn(


[LIB] Running compiler driver in /home/user/oqd-catalyst/scripts/oqd_circuit_qec
[SYSTEM] /home/user/oqd-catalyst/frontend/catalyst/utils/../../../mlir/build/bin/catalyst -o /home/user/oqd-catalyst/scripts/oqd_circuit_qec/oqd_circuit_qec.ll --module-name oqd_circuit_qec --workspace /home/user/oqd-catalyst/scripts/oqd_circuit_qec -verify-each=false --catalyst-pipeline DeviceAgnosticPipeline(quantum-compilation-stage;hlo-lowering-stage;gradient-lowering-stage;bufferization-stage),IonDecompositionStage(func.func(ions-decomposition);func.func(merge-rotations)),IonDialectLoweringStage(func.func(gates-to-pulses{device-toml-loc=/home/user/oqd-catalyst/scripts/calibration_data/device.toml qubit-toml-loc=/home/user/oqd-catalyst/scripts/calibration_data/qubit.toml gate-to-pulse-toml-loc=/home/user/oqd-catalyst/scripts/calibration_data/gate.toml})),IonToLLVMDialectConversion(convert-ion-to-llvm),MLIRToLLVMDialectConversion(llvm-dialect-lowering-stage), --keep-intermediate --verbose /home/user/oqd

In [2]:
import json

print(qml.draw(oqd_circuit_qec)())

with open(compile_results / "oqd_circuit_qec.draw.txt", "w") as f:
    f.write(qml.draw(oqd_circuit_qec)())


QJIT_CIRCUIT()

print(json.dumps(json.load(open(compile_results / openapl_file_name)), indent=2))

 0: ────╭||────╭||────╭||──────────╭||──────────╭||─────────────╭||────╭||──────────────┤  Counts
 1: ──H─├||────├||────├||─╭X───────├||──────────├||───────╭●────├||────├||──H────┤↗├──H─┤        
 2: ────├||──H─├||─╭●─├||─╰●───────├||──────────├||───────│─────├||──H─├||──┤↗├─────────┤        
 3: ──H─├||────├||─╰X─├||───────╭●─├||──────────├||────╭X─│─────├||────├||──H────┤↗├──H─┤        
 4: ────├||────├||────├||───────│──├||──────────├||────│──│─────├||────├||──────────────┤        
 5: ──H─├||────├||────├||───────│──├||────╭X────├||────│──│──╭●─├||────├||──H────┤↗├──H─┤        
 6: ────├||────├||────├||───────│──├||────│─────├||────│──│──│──├||────├||──────────────┤        
 7: ────├||────├||────├||───────│──├||────│─────├||────│──│──│──├||────├||──────────────┤        
 8: ──H─├||────├||────├||────╭●─│──├||────│──╭●─├||─╭X─│──│──│──├||────├||──H────┤↗├──H─┤        
 9: ────├||────├||─╭X─├||────│──╰X─├||────│──╰X─├||─│──│──╰X─│──├||────├||──┤↗├─────────┤        
10: ──H─├||────├||─╰

{
  "class_": "AtomicCircuit",
  "protocol": {
    "class_": "SequentialProtocol",
    "sequence": [
      {
        "class_": "ParallelProtocol",
        "sequence": [
          {
            "beam": {
              "class_": "Beam",
              "detuning": {
                "class_": "MathNum",
                "value": 208570336271826.38
              },
              "phase": {
                "class_": "MathNum",
                "value": 0.0
              },
              "polarization": [
                1,
                0,
                0
              ],
              "rabi": {
                "class_": "MathNum",
                "value": 6283185307.179586
              },
              "target": 1,
              "transition": {
                "class_": "Transition",
                "einsteinA": 41050903.119868636,
                "label": "downstate->estate",
                "level1": {
                  "class_": "Level",
                  "energy": 0.0,
               

/home/user/oqd-catalyst/.venv/lib/python3.12/site-packages/pennylane_qiskit/converter.py:585: UserWarning: pennylane_qiskit.converter: The Reset instruction is not supported by PennyLane, and has not been added to the template.
  warnings.warn(


In [3]:
from oqd_core.interface.atomic import AtomicCircuit
from oqd_core.compiler.atomic.canonicalize import canonicalize_atomic_circuit_factory
from oqd_compiler_infrastructure import Chain, Post
from oqd_bare_metal.compiler.codegen import AtomicToTestbenchV2
from oqd_bare_metal.compiler.optim import (
    SpectrumCoreRemapping,
    SpectrumPrune,
    SpectrumUnwrapResets,
)
import ast_comments as ast


circuit = AtomicCircuit.model_validate_json(
    json.dumps(json.load(open(compile_results / openapl_file_name)), indent=2)
)


compiler = Chain(
    canonicalize_atomic_circuit_factory(),
    Post(
        AtomicToTestbenchV2(
            device_params="./calibration_data/testbench_params.toml",
        ),
    ),
)
optimization_pass = Chain(
    Post(SpectrumCoreRemapping()),
    Post(SpectrumUnwrapResets()),
    Post(SpectrumPrune()),
)

unopt_artiq_experiment = compiler(circuit)
artiq_experiment = optimization_pass(unopt_artiq_experiment)

print(ast.unparse(ast.fix_missing_locations(artiq_experiment)))

with open(compile_results / "oqd_circuit_qec.artiq.py", "w") as f:
    f.write(ast.unparse(ast.fix_missing_locations(artiq_experiment)))

import numpy as np
from artiq.experiment import *

class TestbenchV2Experiment(EnvExperiment):

    @rpc(flags={'async'})
    def transfer_data(self, key, index, value):
        self.mutate_dataset(key=key, index=index, value=value)

    def build(self):
        self.setattr_device('core')
        self.setattr_device('awg')
        self.setattr_device('ttl0')
        self.setattr_device('ttl4')
        self.setattr_device('ttl6')
        self.setattr_device('ttl7')
        self.setattr_device('urukul0_ch0')
        self.setattr_device('urukul0_ch1')
        self.setattr_device('urukul0_ch2')
        self.setattr_device('urukul0_ch3')
        self.setattr_device('urukul1_ch0')
        self.setattr_device('urukul1_ch1')
        self.setattr_device('urukul1_ch2')
        self.setattr_device('urukul1_ch3')

    def program_awg(self):
        # init Spectrum AWG
        self.awg.start()
        self.awg.card_mode('dds')
        self.awg.channel_enable_out(True)
        self.awg.channels_out

In [4]:
# from oqd_core.interface.atomic import AtomicCircuit
# from oqd_core.compiler.atomic.canonicalize import canonicalize_atomic_circuit_factory
# from oqd_compiler_infrastructure import Chain, Post
# from oqd_bare_metal.compiler.codegen import AtomicToBloodstoneV1
# from oqd_bare_metal.compiler.optim import (
#     SpectrumCoreRemapping,
#     SpectrumPrune,
#     SpectrumUnwrapResets,
# )
# import ast_comments as ast


# circuit = AtomicCircuit.model_validate_json(
#     json.dumps(json.load(open(compile_results / openapl_file_name)), indent=2)
# )


# compiler = Chain(
#     canonicalize_atomic_circuit_factory(),
#     Post(
#         AtomicToBloodstoneV1(device_params="./calibration_data/bloodstone_params.toml")
#     ),
# )
# optimization_pass = Chain(
#     Post(SpectrumCoreRemapping(device="raman_awg")),
#     Post(SpectrumUnwrapResets()),
#     Post(SpectrumPrune()),
# )

# unopt_artiq_experiment = compiler(circuit)
# artiq_experiment = optimization_pass(unopt_artiq_experiment)


# with open(compile_results / "oqd_circuit.artiq.py", "w") as f:
#     f.write(ast.unparse(ast.fix_missing_locations(artiq_experiment)))